# Tokenization: from a toy vocabulary to byte-level BPE

Before a model can assign vectors to text, it needs a consistent way to divide text into tokens and map each token to an ID. This CPU-only notebook starts with a deliberately small whitespace tokenizer, including vocabulary IDs and control tokens, before inspecting the byte-level BPE encoding provided by `tiktoken`.

Token IDs are addresses, not semantic quantities. An embedding layer later maps each address to a learnable vector.


In [2]:
import tiktoken

TEXT = 'This text is ready to be tokenized.'
SPECIAL_TOKENS = ('<|unk|>', '<|bos|>', '<|eos|>')


## Build a deliberately limited tokenizer

This tokenizer lowercases text, removes periods, and splits on whitespace. Its constraints are intentional: they make token boundaries, a closed vocabulary, and the unknown-token case easy to inspect before introducing a learned subword tokenizer.


In [3]:
def whitespace_tokens(text: str) -> list[str]:
    return text.lower().replace('.', '').split()

def build_vocabulary(texts: list[str]) -> dict[str, int]:
    words = sorted({token for text in texts for token in whitespace_tokens(text)})
    tokens = list(SPECIAL_TOKENS) + words
    return {token: index for index, token in enumerate(tokens)}

def encode(text: str, vocabulary: dict[str, int]) -> list[int]:
    unknown = vocabulary['<|unk|>']
    return [vocabulary['<|bos|>']] + [vocabulary.get(token, unknown) for token in whitespace_tokens(text)] + [vocabulary['<|eos|>']]

def token_strings_for_ids(ids: list[int], vocabulary: dict[str, int]) -> list[str]:
    inverse = {index: token for token, index in vocabulary.items()}
    return [inverse[index] for index in ids]


## Encode text with vocabulary IDs and control tokens

The vocabulary is built only from `TEXT`, so the new word `input` has no ordinary ID. The encoding marks that position with `<|unk|>` and adds beginning- and end-of-sequence markers, then maps the IDs back to their token strings for inspection.


In [4]:
vocabulary = build_vocabulary([TEXT])
ids = encode('This text input is ready to be tokenized.', vocabulary)
token_strings = token_strings_for_ids(ids, vocabulary)
print(vocabulary)
print(ids)
print(token_strings)
assert token_strings[3] == '<|unk|>'


{'<|unk|>': 0, '<|bos|>': 1, '<|eos|>': 2, 'be': 3, 'is': 4, 'ready': 5, 'text': 6, 'this': 7, 'to': 8, 'tokenized': 9}
[1, 7, 6, 0, 4, 5, 8, 3, 9, 2]
['<|bos|>', 'this', 'text', '<|unk|>', 'is', 'ready', 'to', 'be', 'tokenized', '<|eos|>']


The integer IDs are addresses, not semantic quantities. Changing their order changes no token meaning. `<|bos|>` and `<|eos|>` are control markers rather than ordinary text tokens. `<|unk|>` makes a closed toy vocabulary explicit. Byte-level BPE can instead represent arbitrary text through byte sequences, even when a token does not align with a complete Unicode character.


## Inspect byte-level BPE tokens

Byte Pair Encoding learns frequent merges of adjacent units into a vocabulary of subword pieces. In byte-level BPE, every string is first representable as bytes, so unfamiliar text can be encoded without this toy tokenizer's separate unknown-word token. Individual token pieces may not be valid Unicode on their own.


In [5]:
encoding = tiktoken.get_encoding('cl100k_base')
examples = ['This text input is ready to be tokenized.', 'antidisestablishmentarianism', 'naïve café', '🚀']

for example in examples:
    token_ids = encoding.encode(example)
    token_bytes = [encoding.decode_single_token_bytes(token_id) for token_id in token_ids]
    safe_pieces = [piece.decode('utf-8', errors='replace') for piece in token_bytes]
    print(f'{example!r}: {token_ids}')
    print(f'  bytes: {token_bytes!r}')
    print(f'  display: {safe_pieces}')
    assert encoding.decode(token_ids) == example


'This text input is ready to be tokenized.': [2028, 1495, 1988, 374, 5644, 311, 387, 4037, 1534, 13]
  bytes: [b'This', b' text', b' input', b' is', b' ready', b' to', b' be', b' token', b'ized', b'.']
  display: ['This', ' text', ' input', ' is', ' ready', ' to', ' be', ' token', 'ized', '.']
'antidisestablishmentarianism': [519, 85342, 34500, 479, 8997, 2191]
  bytes: [b'ant', b'idis', b'establish', b'ment', b'arian', b'ism']
  display: ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']
'naïve café': [3458, 38672, 588, 53050]
  bytes: [b'na', b'\xc3\xaf', b've', b' caf\xc3\xa9']
  display: ['na', 'ï', 've', ' café']
'🚀': [9468, 248, 222]
  bytes: [b'\xf0\x9f', b'\x9a', b'\x80']
  display: ['�', '�', '�']


`decode_single_token_bytes()` exposes each byte-level piece safely. An emoji can be split across tokens, so an individual piece may not be valid Unicode on its own and is shown with a replacement character above. Decoding the complete token sequence reconstructs the original text, as the assertions verify.
